In [1]:
# Pad image to squares by adding black pixels, and using a padding that take into account size differences between images.
 

In [3]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))
from config1 import CLASSIFICATION_DATA


In [ ]:
import os
from PIL import Image
import pandas as pd
import numpy as np

In [ ]:
p_df = CLASSIFICATION_DATA / "1.IMAGES_classification"
p_df_augm = CLASSIFICATION_DATA / "IMAGES_augmented"

In [4]:
p_df_padded = CLASSIFICATION_DATA / "IMAGES_padded"
p_df_augm_padded = CLASSIFICATION_DATA / "IMAGES_augmented_padded"



In [7]:

results = []

for img_path in p_df.rglob("*"):
    if img_path.suffix.lower() == ".jpg":
        try:
            with Image.open(img_path) as img:
                width, height = img.size

            results.append({
                "image_name": img_path.name,
                "largest": max(width, height),
                "smallest": min(width, height)
            })

        except Exception as e:
            print(f"Could not read {img_path}: {e}")

df = pd.DataFrame(results)

df.head()

,image_name,largest,smallest
0,ref_cl25_img60.jpg,204,194
1,ref_cl25_img144.jpg,190,180
2,ref_cl25_img176.jpg,198,194
3,ref_cl25_img54.jpg,210,200
4,ref_cl25_img55.jpg,220,204


In [8]:
def z_transform(x, center=250, scale=0.015, max_z=50):
    """
    input x : image size (largest side)
    scale : slope of the sigmoid function
    max_z : max of percentage added on the small images
    output : percentage of black that this image should have on its largest side
    """
    x = np.array(x, dtype=float)
    sigmoid = 1 / (1 + np.exp(scale * (x - center)))
    perc_noir = max_z * sigmoid 
    return perc_noir


In [9]:
def padding_pixel(largest_img_i, perc_black):
    """
    input : lagest side of the initial image
    percentage of black we want to add on this image
    output = nb of pixel to add to get that percentage of black
    """
    nb_black_pixel = largest_img_i*perc_black/100
    return nb_black_pixel


In [10]:
df["perc_black"] = [z_transform(el) for el in list(df["largest"])]
df["padd_to_add"] = [float(padding_pixel(el, df["perc_black"][e])) for e,el in enumerate(list(df["largest"]))]
df["final_size"] = [int(el + df["padd_to_add"][e]) for e,el in enumerate(list(df["largest"]))]

df=df.sort_values(by="largest")
df

,image_name,largest,smallest,perc_black,padd_to_add,final_size
1359,env_cl29_img511.jpg,102,102,4.510156e+01,4.600359e+01,148
367,env_cl29_img970.jpg,102,100,4.510156e+01,4.600359e+01,148
1044,env_cl29_img1074.jpg,104,102,4.496740e+01,4.676609e+01,150
1458,env_cl29_img471.jpg,106,102,4.482998e+01,4.751978e+01,153
1297,env_cl29_img974.jpg,106,92,4.482998e+01,4.751978e+01,153
...,...,...,...,...,...,...
1827,ref_cl29_img23.jpg,1532,348,2.225805e-07,3.409933e-06,1532
23595,env_cl33_img278.jpg,1588,940,9.609034e-08,1.525915e-06,1588
23150,ref_cl33_img274.jpg,1600,720,8.026140e-08,1.284182e-06,1600
866,ref_cl29_img19.jpg,1680,1098,2.417427e-08,4.061277e-07,1680


### Implementation of padding with sigmoid transformation

In [11]:

def add_square_pad(image, padded_image, height, width):
    if height < width:
        padding_i = (width - height) // 2
        padded_image.paste(image, (0, padding_i))
    else:
        padding_j = (height - width) // 2
        padded_image.paste(image, (padding_j, 0))
    return padded_image

def black_padd_size(pin, pout):
    max_size_df=1830
    pad_color = 0 
    for root, _, files in os.walk(pin):
        for file in files:
            if file.lower().endswith('.jpg'):
                img_path = os.path.join(root, file)
                img = Image.open(img_path)
                width, height = img.size
                max_side_i=max(width, height)
                if max_side_i<max_size_df:
                    perc_black = z_transform(max_side_i)
                    nb_black_pixel_toadd = padding_pixel(max_side_i, perc_black)
                    targetsize = max_side_i+nb_black_pixel_toadd
                    padded_image = Image.new('L', (int(targetsize), int(targetsize)), pad_color)
                    padding_leftright = (targetsize - width) // 2
                    padding_topbottom = (targetsize - height) // 2
                    padded_image.paste(img, (int(padding_leftright), int(padding_topbottom)))    
                else: 
                    padded_image = Image.new('L', (max_side_i,max_side_i), pad_color)
                    padded_image=add_square_pad(img, padded_image, height, width)
                folder_vi_p = img_path.replace(pin, "")
                output_dir = pout + folder_vi_p.replace(file, "")
                os.makedirs(output_dir, exist_ok=True)
                padded_image.save(output_dir + file)
                img.close()



In [ ]:

implementation = "no" 
if implementation == "yes":
    black_padd_size(str(p_df), str(p_df_padded))
    black_padd_size(str(p_df_augm), str(p_df_augm_padded))
